In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import sys
from logging import INFO, StreamHandler, getLogger

logger = getLogger()
if not logger.hasHandlers():
    logger.addHandler(StreamHandler(sys.stdout))
logger.setLevel(INFO)

In [ ]:
import os
import pathlib
from itertools import product

import numpy as np
import xarray as xr

ROOT_DIR = str(pathlib.Path(".").resolve().parent.parent)
print(f"ROOT_DIR: {ROOT_DIR}")

In [ ]:
sys.path.append(ROOT_DIR)

In [ ]:
from src.cesm_le.multi_process_helper import (
    parallel_process_dataset_with_process_worker,
)

In [ ]:
CESM_DATA_DIR = f"{ROOT_DIR}/data/cesm_le"
NUM_PROCESSES = 64

In [ ]:
PROCESSED_DATA_DIR = f"{ROOT_DIR}/data/processed"
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

# Metrics over time


In [ ]:
for remove_annual_cycle, path, (ens_start, ens_end) in product(
    ["remove_climatology"],
    [
        f"{CESM_DATA_DIR}/atm-monthly-cesmLE-20C-TREFHT.nc",
        f"{CESM_DATA_DIR}/atm-monthly-cesmLE-RCP85-TREFHT.nc",
        f"{CESM_DATA_DIR}/atm-monthly-cesmLE-20C-PRECT.nc",
        f"{CESM_DATA_DIR}/atm-monthly-cesmLE-RCP85-PRECT.nc",
    ],
    [(0, 20), (20, 40), (0, 40)],
):
    try:
        if ens_start == 0 and ens_end == 40:
            pass
        else:
            if remove_annual_cycle != "remove_climatology":
                continue

        assert os.path.exists(path), f"File not found: {path}"

        name = os.path.basename(path).replace(".nc", "")
        if remove_annual_cycle == "remove_climatology":
            out_path = f"{PROCESSED_DATA_DIR}/{name}_Ens{ens_start:02}-{ens_end:02}_no_annual_cycle_climatology.npz"
        elif remove_annual_cycle == "remove_running_mean":
            out_path = f"{PROCESSED_DATA_DIR}/{name}_Ens{ens_start:02}-{ens_end:02}_no_annual_cycle_running_mean.npz"
        elif remove_annual_cycle == "remove_annual_harmonic":
            out_path = f"{PROCESSED_DATA_DIR}/{name}_Ens{ens_start:02}-{ens_end:02}_no_annual_cycle_annual_harmonic.npz"
        elif remove_annual_cycle == "none":
            out_path = f"{PROCESSED_DATA_DIR}/{name}_Ens{ens_start:02}-{ens_end:02}.npz"
        else:
            raise Exception(f"Unknown remove_annual_cycle: {remove_annual_cycle}")

        if "TREFHT" in path:
            var_name = "TREFHT"
            ave_nbins = 57 if "-20C-" in path else 47
            # ave_nbins below is tuned specifically for the remove_climatology
            # case; it has not been re-tuned for remove_running_mean.
            if (
                remove_annual_cycle == "remove_climatology"
                or remove_annual_cycle == "remove_running_mean"
                or remove_annual_cycle == "remove_annual_harmonic"
            ):
                ave_nbins = 50 if "-20C-" in path else 46

        elif "PRECT" in path:
            var_name = "PRECT"
            ave_nbins = 108 if "-20C-" in path else 119
            if (
                remove_annual_cycle == "remove_climatology"
                or remove_annual_cycle == "remove_running_mean"
                or remove_annual_cycle == "remove_annual_harmonic"
            ):
                ave_nbins = 97 if "-20C-" in path else 106
        else:
            raise ValueError(f"Unknown variable in path: {path}")

        if os.path.exists(out_path):
            print(f"Skip because the file already exists: {out_path}")
        else:
            print(f"{path=}, {ave_nbins=}, {out_path=}")
            (
                g_std,
                g_w,
                g_w_sort,
                g_sane,
                nbins,
                g_sane_ave_nbis,
                g_sane_original,
                g_w_sort_tot,
            ) = parallel_process_dataset_with_process_worker(
                ds=xr.open_dataset(path),
                n_processes=NUM_PROCESSES,
                ens_start=ens_start,
                ens_end=ens_end,
                var_name=var_name,
                max_nbins=500,
                ave_nbins=ave_nbins,
                use_logscale=False,
                remove_annual_cycle=remove_annual_cycle,
            )
            np.savez(
                out_path,
                g_std=g_std,
                g_sane=g_sane,
                g_w=g_w,
                g_w_sort=g_w_sort,
                nbins=nbins,
                g_sane_ave_nbis=g_sane_ave_nbis,
                g_sane_original=g_sane_original,
                g_w_sort_tot=g_w_sort_tot,
            )
            print(f"Saved to {out_path}")
    except Exception as e:
        print(f"{path=}, {ens_start=}, {ens_end}")
        print(e)